# Performance Analysis

Here we optimize method hyperparameters on simulated examples and examine method accuracy and bias.

In [1]:
import multiprocessing as mp
import pandas, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import re, os, glob
from tqdm import tqdm
from collections import defaultdict
from warnings import catch_warnings
#mp.set_start_method("fork", force=True) # if running on mac, uncomment this

from pynumdiff.utils.simulate import sine, triangle, pop_dyn, linear_autonomous, pi_cruise_control, lorenz_x
from pynumdiff.utils.evaluate import rmse, error_correlation
from pynumdiff.finite_difference import finitediff
from pynumdiff.smooth_finite_difference import kerneldiff, butterdiff
from pynumdiff.polynomial_fit import splinediff, polydiff, savgoldiff
from pynumdiff.basis_fit import spectraldiff, rbfdiff, waveletdiff
from pynumdiff.total_variation_regularization import tvrdiff, smooth_acceleration
from pynumdiff.kalman_smooth import rtsdiff, robustdiff
from pynumdiff.linear_model import lineardiff
from pynumdiff.optimize import optimize

Experimental search space parameters. There are many random seeds to ensure results are not flukes.

In [2]:
random_seeds = [1, 7, 31, 45, 57, 101, 256, 343, 737, 1024, 10000]# 11 of these
random_seeds += [1000001 + i for i in range(22)] # 22 more
random_seeds += [2000001 + i for i in range(19)] # 19 more
print(len(random_seeds))
dts = [0.005, 0.01, 0.02, 0.04] #, 0.08]
noise_types = [('normal', [0, 0.1]), ('laplace', [0, 0.1]), ('uniform', [-0.2, 0.2])]
noise_scales = [0.5, 1, 2, 4]
bandlimits = [1, 2, 3, 4, 5] # high frequency of signal in the data.

52


In [3]:
methods = [(kerneldiff, 'KernelDiff'),
			(butterdiff, 'ButterDiff'),
			(finitediff, 'IteratedFD'),
			(polydiff, 'PolyDiff'),
			(savgoldiff, 'SavGolDiff'),
			(splinediff, 'SplineDiff'),
			(spectraldiff, 'SpectralDiff'),
			(rbfdiff, 'RBFDiff'),
			(waveletdiff, 'WaveletDiff'),
			(tvrdiff, 'TVRDiff'),
			(smooth_acceleration, 'SmoothAccelTVR'),
			(rtsdiff, 'RTSDiff'),
			(robustdiff, 'RobustDiff'),
			(lineardiff, 'LinearDiff')]
sims = [(pi_cruise_control, 'Cruise Control'),
		(sine, 'Sum of Sines'),
		(triangle, 'Triangles'),
		(pop_dyn, 'Logistic Growth'),
		(linear_autonomous, 'Linear Autonomous'),
		(lorenz_x, 'Lorenz First Dimension')]

## Experiment

In [4]:
def one_sim(dt, bandlimit, noise_type, noise_params, noise_scale, outliers, random_seed, sname, mnames):
	"""Run the methods named in `mnames` against a sim and return the column of results, a reasonable work chunk size so
	cores stay busy to the end instead of trailing off while a few long jobs finish; the parent then merges into one file
	per configuration. Taking a subset means a newly added method can be filled in without recomputing the others."""
	sim = next(func for func,name in sims if name == sname) # reverse lookup by name, because you can't pass function directly and remain picklable

	x, x_truth, dxdt_truth = sim(duration=4, dt=dt, noise_type=noise_type,
								noise_parameters=noise_scale*np.array(noise_params),
								outliers=outliers, random_seed=random_seed)

	column = {} # or a partial column, possibly; only new values written
	for method,mname in methods:
		if mname not in mnames: continue # only compute what this config still lacks
		# alternatively, pass dxdt_truth=dxdt_truth and metric='rmse' or 'error_correlation' instead of bandlimit to target known truth
		best_params, best_score = optimize(method, x, dt, bandlimit=bandlimit, parallel=False, huberM=(2 if outliers else 6),
			search_space_updates=({'order':{1,2,3}} if method == tvrdiff and sim == triangle else {})) # so TVR can fit straight lines to the triangles
		with catch_warnings(action="ignore", category=UserWarning): # the call itself can print warnings, but param choices are verified by this point
			x_hat, dxdt_hat = method(x, dt, **best_params)

		rmse_dxdt = rmse(dxdt_truth, dxdt_hat)
		ec = error_correlation(dxdt_truth, dxdt_hat)
		column[mname] = f"RMSE: {rmse_dxdt:.5g}<br/>R^2: {ec:.5g}"

	return f"{bandlimit}_{dt}_{noise_type}_{noise_scale}_{outliers}_{random_seed}", sname, column


SyntaxError: unmatched ')' (1358642043.py, line 16)

The hypercube was getting too dang big, so take intentional slices to answer specific questions.

In [ ]:
jobs = []
for random_seed in random_seeds:
	# Does dt matter? Vary dt, keeping everything else constant
	jobs += [{'dt':dt, 'bandlimit':3, 'noise_type':'normal', 'noise_params':[0, 0.1],
				'noise_scale':1, 'outliers':False, 'random_seed':random_seed} for dt in dts]

	# Does noise type matter? Vary noise type, keeping everything else constant
	jobs += [{'dt':0.01, 'bandlimit':3, 'noise_type':noise_type, 'noise_params':noise_params,
				'noise_scale':1, 'outliers':False, 'random_seed':random_seed} for noise_type,noise_params in noise_types
				if noise_type != 'normal']
	
	# Does noise scale matter? Vary noise scale, keeping everything else constant
	jobs += [{'dt':0.01, 'bandlimit':3, 'noise_type':'normal', 'noise_params':[0, 0.1],
				'noise_scale':noise_scale, 'outliers':False, 'random_seed':random_seed} for noise_scale in noise_scales
				if noise_scale != 1]
	
	# Does the presence of outliers matter? Vary presence of outliers, keeping everything else constant
	jobs += [{'dt':0.01, 'bandlimit':3, 'noise_type':'normal', 'noise_params':[0, 0.1],
				'noise_scale':1, 'outliers':outliers, 'random_seed':random_seed} for outliers in [True]]
	
	# Does the bandlimit matter? Vary bandlimit, keeping everything else constant
	jobs += [{'dt':0.01, 'bandlimit':bandlimit, 'noise_type':'normal', 'noise_params':[0, 0.1],
				'noise_scale':1, 'outliers':False, 'random_seed':random_seed} for bandlimit in bandlimits
				if bandlimit != 3]

# Fan out over (config, sim) to keep work chunks reasonable,
for j in jobs: # skipping already-computed answers, as identified by (config, sim, method).
	try:
		done = set(pandas.read_csv(f"~/Desktop/results/{j['bandlimit']}_{j['dt']}_{j['noise_type']}_{j['noise_scale']}"
									f"_{j['outliers']}_{j['random_seed']}.csv", index_col=0, usecols=[0]).index)
		j['mnames'] = [mname for _,mname in methods if mname not in done]
	except FileNotFoundError: j['mnames'] = [mname for _,mname in methods]
jobs = [j | {'sname': sname} for j in jobs if j['mnames'] for _,sname in sims] # Cartesian product with sims; | creates new dict with one more entry

def run(kwargs): return one_sim(**kwargs)
pending = defaultdict(dict) # config key -> {sim name: column}, written out once all its sims have landed
with mp.Pool() as pool:
	for config,sname,column in tqdm(pool.imap_unordered(run, jobs, chunksize=1), total=len(jobs), ncols=140): # chunk so work isn't pre-split unevenly
		pending[config][sname] = column									# imap_unordered so results aren't held up by earlier-begun tasks

		# Once all the methods are done for a parameter combo, write the results to a single file
		if len(pending[config]) == len(sims):
			try: res = pandas.read_csv(f"~/Desktop/results/{config}.csv", index_col=0) # merge into what is already there
			except FileNotFoundError: res = pandas.DataFrame(index=[x[1] for x in methods], columns=[x[1] for x in sims])
			for sn,col in pending.pop(config).items(): # pop so finished configs don't accumulate for the whole run
				for mn,val in col.items(): res.loc[mn, sn] = val
			res.to_csv(f"~/Desktop/results/{config}.csv")

## Plot Results

In [ ]:
def plot_perf(indie, vals, error_bars_gaps):
	"""
	:param str indie: the name of the independent variable
	:param list vals: values for the independent variable
	:param error_bars_gaps: how much space to give for the central symbol
	"""
	fig, ax = plt.subplots(2, 1, figsize=(30, 12), sharex=True, gridspec_kw={'hspace': 0})

	markers = ['p', 'd', '*', '2', '+', 'x', 'o', '.', r'$w$', r'$\gamma$', r'$\tilde{\gamma}$', '^', 's', '_']
	facecolors = ['none' if i not in [3,4,5,7,13] else None for i in range(len(markers))] # 'none makes transparent'
	sizes = [90, 80, 120, 120, 100, 70, 70, 70, 90, 90, 170, 90, 70, 100]
	cmap = plt.get_cmap('turbo', 6) # Assign a unique color for each simulation
	colors = [cmap(i) for i in range(6)]; colors[0] = 'purple'; colors[-1] = 'red'
	colors[2] = [x*0.8 for x in colors[2]]; colors[3] = mcolors.to_rgb('gold'); colors[3] = [x*0.9 for x in colors[3]]
	colors[4] = [min(1, x*1.2) for x in colors[4]]

	point = {'bandlimit':3, # all slices go through here
		'dt':0.01, 'noise_type':'normal', 'noise_scale':1, 'outliers':False}
	
	all_res = defaultdict(list)
	for random_seed in random_seeds:
		for v in vals:
			try:
				point[indie] = v # replace value
				res = pandas.read_csv(
					f"~/Desktop/results/{point['bandlimit']}_{point['dt']}_{point['noise_type']}_{
						point['noise_scale']}_{point['outliers']}_{random_seed}.csv",
					index_col=0)
	
				for sim,sname in sims:
					for method,mname in methods:
						# extract the numbers
						rmse, ec = map(float, re.search(r"RMSE:\s*([\d.e+-]+)<br/>R\^2:\s*([\d.e+-]+)", res.loc[mname, sname]).groups())
						all_res[(v, mname, sname)].append((rmse, ec))

			except FileNotFoundError:
				print(f"{point['bandlimit']}_{point['dt']}_{point['noise_type']}_{
						point['noise_scale']}_{point['outliers']}_{random_seed}.csv not yet computed")

	for a,v in enumerate(vals):
		for j,(sim,sname) in enumerate(sims):
			for i,(method,mname) in enumerate(methods):				
				for k,perfs in enumerate(zip(*all_res[(v, mname, sname)])):
					mean_perf = np.mean(perfs) # sample mean
					std_perf = np.std(perfs, ddof=1) # sample standard deviation
					#ci_perf = t_crit = t_dist.ppf(1 - 0.05/2, len(perfs)-1) * std_perf / np.sqrt(len(perfs)) # 95% confidence interval
					x = a*(len(sims)+0.2) + j + 0.05 + 0.9*i/len(methods)
					ax[k].scatter(x, mean_perf, color=colors[j], marker=markers[i], s=sizes[i],
								facecolor=facecolors[i], linewidth=1.5, label=mname if a==0 and a==0 and j==0 and k==0 else None) # label only once per method
					ax[k].vlines(x, mean_perf - std_perf, mean_perf - error_bars_gaps[k], color=colors[j], linewidth=1.5)
					ax[k].vlines(x, mean_perf + error_bars_gaps[k], mean_perf + std_perf, color=colors[j], linewidth=1.5)
					#ax[k].scatter((x, x), (mean_perf - ci_perf, mean_perf + ci_perf), color=colors[j], marker='_', s=20, linewidth=1.5)

	for k in range(2):
		for b in range(len(vals)-1):
			ax[k].axvline((len(sims)+0.2)*(b+1)-0.1, color="gray", linestyle="--", alpha=0.5)
		ax[k].tick_params(axis="x", length=0, labelsize=25) # length=0 hides tick lines
		ax[k].set_xlim(-0.1, len(vals)*(len(sims) + 0.2) - 0.1)
		ax[k].tick_params(axis='y', labelsize=18)
		ax[k].set_ylabel([r'RMSE($\mathbf{\hat{\dot{x}}}$, $\mathbf{\dot{x}}$)', r'Corr($\mathbf{\hat{\dot{x}}} - \mathbf{\dot{x}}$, $\mathbf{\dot{x}}$)'][k], fontsize=25)
	ax[0].set_xticklabels([])
	for label in ax[1].get_yticklabels(): label.set_fontstyle('italic')
	ax[0].set_ylim(0, 3)
	ax[1].set_ylim(0, 0.3)
	ax[1].set_xticks([(len(sims)+0.2)*(b+1)-(len(sims)+0.2)/2 - 0.1/2 for b in range(len(vals))]) 
	
	legend1 = ax[0].legend(ncol=2, columnspacing=0.5, handletextpad=0, loc='upper left', fontsize=15)
	ax[0].add_artist(legend1)
	for handle in legend1.legend_handles:
		handle.set_edgecolor('dimgray')
		if len(handle.get_facecolor()) == 1: handle.set_facecolor('dimgray') # for those that are filled
	sim_patches = [mpatches.Patch(color=colors[j], label=sname) for j,(sim,sname) in enumerate(sims)]
	legend2 = ax[0].legend(handles=sim_patches, loc='upper left', fontsize=15, bbox_to_anchor=(0.175, 1.0))

	return fig, ax


In [ ]:
fig, ax = plot_perf('outliers', [False, True], (0.09, 0.005))
ax[0].set_ylim(0, 3)
ax[1].set_ylim(0, 0.25)
ax[1].set_yticks(ax[1].get_yticks()[:-1])
ax[1].set_xticklabels(["no outliers", "with 1% of data being outliers"])
fig.suptitle(r"Outliers' Effect on RMSE and Error Correlation", fontsize=32, y=0.92)
fig.savefig(os.path.expanduser("~/Desktop/vary_outliers.png"), bbox_inches='tight')

In [ ]:
fig, ax = plot_perf('noise_type', [x[0] for x in noise_types], (0.05, 0.005))
ax[0].set_ylim(0, 2.2)
ax[1].set_ylim(0, 0.25)
ax[1].set_yticks(ax[1].get_yticks()[:-1])
ax[1].set_xticklabels([rf"{x[0]}" for x in noise_types])
fig.suptitle(r"Noise Type's Effect on RMSE and Error Correlaion", fontsize=32, y=0.92)
fig.savefig(os.path.expanduser("~/Desktop/vary_noise_type.png"), bbox_inches='tight')

In [ ]:
fig, ax = plot_perf('noise_scale', noise_scales, (0.07, 0.005))
ax[0].set_ylim(0, 3)
ax[1].set_ylim(0, 0.25)
ax[1].set_yticks(ax[1].get_yticks()[:-1])
ax[1].set_xticklabels([rf"scale={x}" for x in noise_scales])
fig.suptitle(r"Noise Scale's Effect on RMSE and Error Correlation", fontsize=32, y=0.92)
fig.savefig(os.path.expanduser("~/Desktop/vary_noise_scale.png"), bbox_inches='tight')

In [ ]:
fig, ax = plot_perf('dt', dts, (0.07, 0.005))
ax[1].set_xticklabels([rf"$\Delta t$={dt}" for dt in dts])
ax[1].set_ylim(0, 0.25)
ax[1].set_yticks(ax[1].get_yticks()[:-1])
fig.suptitle(r"$\Delta t$'s Effect on RMSE and Error Correlation", fontsize=32, y=0.92)
fig.savefig(os.path.expanduser("~/Desktop/vary_dt.png"), bbox_inches='tight')

In [ ]:
fig, ax = plot_perf('bandlimit', bandlimits, (0.05, 0.005))
ax[0].set_ylim(0, 2.4)
ax[1].set_ylim(0, 0.65)
ax[1].set_xticklabels([rf"{f} Hz, $\gamma\!\approx\!${np.exp(-1.6*np.log(f) - 0.71*np.log(0.01) - 5.1):.2g}" for f in bandlimits])
fig.suptitle(r"Bandlimit's Effect on RMSE and Error Correlation", fontsize=32, y=0.92)
fig.savefig(os.path.expanduser("~/Desktop/vary_bandlimit.png"), bbox_inches='tight')

## Comparing Methods Within a Cell

The plots above answer "how does performance move as one condition changes." They do not answer "which method should I reach for," because raw RMSE cannot be averaged across cells: it carries the units of the derivative and its scale varies by an order of magnitude between a quiet configuration and a noisy one, so a median of raw RMSE reports which *configurations* were hard, not which method was good.

Two statistics avoid this without any rescaling, because both are computed inside a cell where all fourteen methods saw byte-identical data:

**Rank.** Sort the methods by RMSE within the cell; a method's rank is its position, 1 for best and 14 for worst. Scale-free by construction and unmoved by how extreme the errors happen to be. *Mean rank* answers "where does this method usually stand." Its blind spot is margin: finishing second by 0.1% and second by 50% score the same.

**Relative RMSE.** This method's RMSE divided by the smallest RMSE in that cell. Also scale-free, and it keeps exactly the margins rank discards: 1.00 means it won, 1.30 means it finished 30% worse than whatever did. Its blind spot is that the denominator is the field, so adding or removing a strong method moves every number.

Two more read the tails of the rank distribution. *Top-3 rate* is how often a method is among the best available. *Bottom-3 rate* is how often someone using it would be badly served, which is the statistic that matters for a default recommendation, since a default is used by people who will not check.

*Win share*, the fraction of cells ranked first, is the most interpretable and the most blind: a method that finishes second by 1% everywhere scores zero.

In [ ]:
def load(folder="results"):
	"""Parse a results directory into one row per (config, sim, method), for statistics that need a whole cell."""
	rows = []
	for path in glob.glob(os.path.expanduser(f"~/Desktop/{folder}/*.csv")):
		cf, dt, nt, ns, ol, seed = os.path.basename(path)[:-4].split('_')
		table = pandas.read_csv(path, index_col=0)
		for mname in table.index:
			for sname in table.columns:
				cell = table.loc[mname, sname]
				if not isinstance(cell, str): continue # a method that has not been filled in yet
				r, ec = map(float, re.search(r"RMSE:\s*([\d.e+-]+)<br/>R\^2:\s*([\d.e+-]+)", cell).groups())
				rows.append((float(cf), float(dt), nt, float(ns), ol=='True', int(seed), mname, sname, r, ec))
	return pandas.DataFrame(rows, columns=['cf','dt','noise','ns','outliers','seed','method','sim','rmse','ec'])

key = ['cf','dt','noise','ns','outliers','seed','sim'] # identifies one cell, in which every method saw identical data

def method_stats(df):
	"""Scale-free per-method statistics, each computed within a cell where every method saw identical data."""
	g = df.groupby(key).rmse
	d = df.assign(rel=df.rmse/g.transform('min'), rank=g.rank(method='average'))
	return pandas.DataFrame({
		'win %':      d.assign(v=d['rank'] == 1).groupby('method').v.mean()*100,
		'top-3 %':    d.assign(v=d['rank'] <= 3).groupby('method').v.mean()*100,
		'mean rank':  d.groupby('method')['rank'].mean(),
		'bottom-3 %': d.assign(v=d['rank'] >= 12).groupby('method').v.mean()*100,
		'rel rmse':   d.groupby('method').rel.median(),
		'rel p90':    d.groupby('method').rel.quantile(0.90)})

stats = method_stats(load())
print(stats.sort_values('mean rank').round(2).to_string())

print(f"\nSpearman(win %, mean rank)    = {stats['win %'].corr(stats['mean rank'], method='spearman'):+.2f}")
print(f"Spearman(win %, rel rmse)     = {stats['win %'].corr(stats['rel rmse'], method='spearman'):+.2f}")
print(f"Spearman(mean rank, rel rmse) = {stats['mean rank'].corr(stats['rel rmse'], method='spearman'):+.2f}")